# TRL GRPOTrainer single-turn verifiable reward

**Objective:** single-turn deterministic scoring; study reward ties, advantage and clipping; do NOT represent as Harbor multi-turn RL.

This is a guided training starter. Training is deliberately **not executed** until you complete the numbered checks below. Prerequisite: [GRPO mechanics lab](01_grpo_clipping_diagnostics.ipynb). Save the checkpoint and metric provenance for the next lab.


In [ ]:
# Colab GPU recommended. Validate TRL version and tokenizer/model requirements.
%pip -q install -U "trl" "transformers" "datasets" "accelerate" "matplotlib"

import json, inspect
import trl
from datasets import Dataset
from trl import GRPOTrainer, GRPOConfig
print("TRL", trl.__version__)
print("Trainer constructor:", inspect.signature(GRPOTrainer))
print("Config constructor:", inspect.signature(GRPOConfig))

# TODO(student): choose a small compatible causal LM/checkpoint based on actual GPU memory.
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"


In [ ]:
# Intentionally verifiable SINGLE-TURN toy reward, not Harbor agent RL.
dataset = Dataset.from_list([
    {"prompt": [{"role": "user", "content": "Return exactly the word blue."}]},
    {"prompt": [{"role": "user", "content": "Return exactly the word green."}]},
])
def exact_color_reward(completions, **kwargs):
    # TODO(student): verify this signature against installed TRL and reward test fixtures.
    allowed = {"blue", "green"}
    texts = [c[0]["content"] if isinstance(c, list) else str(c) for c in completions]
    return [1.0 if t.strip().lower() in allowed else 0.0 for t in texts]

config = GRPOConfig(
    output_dir="artifacts/grpo-single-turn",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    num_generations=2,
    max_completion_length=64,
    max_prompt_length=128,
    logging_steps=1,
    report_to="none",
)
# TODO(student): confirm effective generation batch >= num_generations, reward-group diversity,
# and the exact semantics of sampled-completion reward for your task.
# trainer = GRPOTrainer(model=MODEL_ID, args=config, train_dataset=dataset,
#                       reward_funcs=exact_color_reward)
# trainer.train()
# TODO(student): collect ratio clipping, KL, gradient norms, reward ties and trusted eval.


## Diagnostics and plotting assignment
1. Define the numerical quantities and denominators. 2. Extract training/eval measurements. 3. Compare the appropriate checkpoints and controlled baselines. 4. Interpret discrepancies and design next discriminating probe.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

def plot_metric(x, y, title, ylabel, path):
    """Rendering ONLY. Student must implement x/y calculation from real trainer logs."""
    fig, ax = plt.subplots(figsize=(7.2, 4.5), layout="constrained")
    ax.plot(x, y, marker=".", linewidth=1.8)
    ax.set(title=title, xlabel="optimizer step", ylabel=ylabel)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.grid(alpha=0.25)
    fig.savefig(path, dpi=220, bbox_inches="tight")
    plt.show()

# TODO(student): extract metric points with provenance from trainer.state.log_history.
# TODO(student): choose correct step axis, smoothing policy, and diagnostic comparisons.
# plot_metric(steps, values, "Training diagnostic", "Value", "artifacts/diagnostic.png")


## Exit criteria
- Record model revision, TRL version, tokenizer, seed and GPU.
- Confirm split integrity, verifier correctness, and no label/answer leakage.
- Plot at least three diagnostics with student-computed axes and quantities.
- Save an experiment report with measured outcomes versus hypotheses.
- Do not claim Harbor multi-turn GRPO integration from this notebook.
